In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("/content/Quote_dataset.txt")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [5]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [6]:
quotes = quotes.str.lower()

In [7]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [8]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [10]:
vocab_size = 10000

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [11]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [12]:
sequence = tokinizer.texts_to_sequences(quotes)

In [13]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [14]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [15]:
X = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [16]:
len(X)

85271

In [17]:
len(y)

85271

In [18]:
# Applying Padding

max_len = max(len(x) for x in X)
print(max_len)

745


In [19]:
# Here padding applied

from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

In [20]:
# Converted to array

y = np.array(y)

In [21]:
X_padded.shape

(85271, 745)

In [22]:
from tensorflow.keras.utils import to_categorical

# still giving memory error i have updated the code to float 32
# y_one_hot = to_categorical(y, num_classes=vocab_size).astype(np.float32)
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [23]:
y.shape

(85271,)

In [24]:
# y_one_hot.shape

In [25]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [26]:
embedding_dim = 50
rnn_units = 128

In [27]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [28]:
rnn_model.compile(
    optimizer='adam',
    # loss='sparse_categorical_crossentropy',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [29]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [30]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [31]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [32]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [33]:
epochs = 10
batch_size = 128

In [34]:
history_rnn = rnn_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 589s 978ms/step - accuracy: 0.0450 - loss: 6.7229 - val_accuracy: 0.0578 - val_loss: 6.5867
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 623s 978ms/step - accuracy: 0.0730 - loss: 6.1737 - val_accuracy: 0.0857 - val_loss: 6.3363
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 611s 961ms/step - accuracy: 0.1001 - loss: 5.8066 - val_accuracy: 0.0993 - val_loss: 6.2682
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 580s 967ms/step - accuracy: 0.1167 - loss: 5.5066 - val_accuracy: 0.1052 - val_loss: 6.2889
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 644s 1s/step - accuracy: 0.1302 - loss: 5.2470 - val_accuracy: 0.1109 - val_loss: 6.3487
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 599s 967ms/step - accuracy: 0.1424 - loss: 5.0111 - val_accuracy: 0.1103 - val_loss: 6.3788
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 624s 970ms/step - accuracy: 0.1547 - loss: 4.7928 - val_accuracy: 0.1101 - val_loss: 6.4398
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 645s 1s/step - accuracy: 0.1720 - loss:

In [35]:
epochs = 10
batch_size = 128

history_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1468s 2s/step - accuracy: 0.0385 - loss: 6.7476 - val_accuracy: 0.0480 - val_loss: 6.6628
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1451s 2s/step - accuracy: 0.0571 - loss: 6.3303 - val_accuracy: 0.0603 - val_loss: 6.5839
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1444s 2s/step - accuracy: 0.0771 - loss: 6.0775 - val_accuracy: 0.0816 - val_loss: 6.4840
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1444s 2s/step - accuracy: 0.0947 - loss: 5.8584 - val_accuracy: 0.0905 - val_loss: 6.4413
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1428s 2s/step - accuracy: 0.1049 - loss: 5.6787 - val_accuracy: 0.1004 - val_loss: 6.4450
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1493s 2s/step - accuracy: 0.1144 - loss: 5.5083 - val_accuracy: 0.1025 - val_loss: 6.4461
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1446s 2s/step - accuracy: 0.1233 - loss: 5.3499 - val_accuracy: 0.1099 - val_loss: 6.4543
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 1467s 2s/step - accuracy: 0.1315 - loss: 5.2026 - 

In [36]:
from tensorflow.keras.models import load_model

# The model 'lstm_model' should already be available in memory from the previous training cell.
# If you intended to load a previously saved model, ensure 'lstm_model.h5' exists and is saved before this cell.
# lstm_model = load_model("lstm_model.h5")

In [37]:
lstm_model.save("lstm_model.h5")

In [38]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [39]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [40]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)

  # Handle cases where the predicted index is not in the vocabulary
  if pred_index == 0:
    return "<pad>" # Special token for padding
  elif pred_index not in index_to_word:
    return "<unk>" # Unknown word
  else:
    return index_to_word[pred_index]

In [41]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

are


In [42]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [43]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)

are you a  man who was a man who was a man who


In [44]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [45]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)